# MoE-MedIR — Kaggle Training Notebook

> **Paper**: Bridging the Intra-Modal Heterogeneity Gap: Mixture-of-Experts for Multi-Modality Medical Image Retrieval  
> **Conference**: ICARCV 2026

**Setup**: Notebook → Settings → Accelerator → GPU T4/P100  
**Internet**: Settings → Internet → On (cần để clone git)

## Cell 1 — Check GPU

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1),'GB')
else:
    raise RuntimeError('No GPU — Settings → Accelerator → GPU')

## Cell 2 — Install Packages

In [ ]:
%%capture
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'medmnist', 'open_clip_torch', 'timm',
    'pytorch-metric-learning', 'scikit-learn',
    'matplotlib', 'seaborn'], check=True)
print('Done.')

## Cell 3 — Clone Project từ GitHub

**Bước 1**: Tạo repo trên GitHub rồi đổi `REPO_URL` bên dưới.  
**Kaggle**: Settings → Internet phải **On**.

In [ ]:
import os, subprocess

REPO_URL     = 'https://github.com/YOUR_USERNAME/moe_medir.git'  # <-- đổi
PROJECT_ROOT = '/kaggle/working/moe_medir'

if os.path.exists(PROJECT_ROOT):
    subprocess.run(['git', '-C', PROJECT_ROOT, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, PROJECT_ROOT], check=True)

os.chdir(PROJECT_ROOT)
print('Working directory:', os.getcwd())
subprocess.run(['git', 'log', '--oneline', '-3'])

## Cell 4 — Sanity Check

Kiểm tra imports + config + model **trước khi download data**.

In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, 'test_imports.py'], capture_output=False)
if result.returncode != 0:
    print('\n[!] Fix lỗi trên trước khi tiếp tục!')
else:
    print('\nAll checks passed — safe to proceed.')

## Cell 5 — (Optional) Load Pre-Extracted Features

Nếu đã upload features thành Kaggle Dataset riêng:

In [ ]:
import os, shutil
FEATURES_INPUT = '/kaggle/input/moe-medir-features'  # đổi nếu có
if os.path.exists(FEATURES_INPUT):
    dest = 'data/features'
    os.makedirs(dest, exist_ok=True)
    shutil.copytree(FEATURES_INPUT, dest, dirs_exist_ok=True)
    npy = [f for f in os.listdir(dest) if f.endswith('.npy')]
    print(f'Loaded {len(npy)} .npy files.')
else:
    print('No pre-extracted features — will extract in Cell 6.')

## Cell 6 — Extract Features

In [ ]:
import os, subprocess, sys
if os.path.exists('data/features/pathmnist_train_feat.npy'):
    print('Features already extracted — skipping.')
else:
    subprocess.run([sys.executable, 'extract_features.py'], check=True)

## Cell 7 — Zero-Shot Baselines

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'baselines/zeroshot.py'], check=True)

## Cell 8 — HashNet Baseline

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'baselines/hashnet.py'], check=True)

## Cell 9 — Linear Baseline

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'train.py', '--model', 'linear', '--epochs', '50'], check=True)
subprocess.run([sys.executable, 'eval/evaluate.py', '--model', 'linear'], check=True)

## Cell 10 — MLP Baseline

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'train.py', '--model', 'mlp', '--epochs', '50'], check=True)
subprocess.run([sys.executable, 'eval/evaluate.py', '--model', 'mlp'], check=True)

## Cell 11 — MoE-MedIR (Main Model)

~15 phút trên P100. Log `avg mAP@R` mỗi 5 epochs.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'train.py', '--model', 'moe', '--epochs', '50'], check=True)

## Cell 12 — Full Test Evaluation

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'eval/evaluate.py', '--model', 'moe'], check=True)

## Cell 13 — Compile Comparison Table

In [ ]:
import subprocess, sys, pandas as pd
subprocess.run([sys.executable, 'baselines/compile_table.py'], check=True)
df = pd.read_csv('results/table_main.csv')
print(df.to_string(index=False))

## Cell 14 — Print LaTeX Table

In [ ]:
with open('results/table_main.tex') as f:
    print(f.read())

## Cell 15 — Expert Routing Heatmap

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'analysis/routing_heatmap.py'], check=True)

## Cell 16 — t-SNE Visualisation

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'analysis/tsne_plot.py', '--model', 'moe', '--max_samples', '400'], check=True)

## Cell 17 — Save Results

Tất cả file trong `/kaggle/working/` tự động lưu — tải về qua tab Output.

In [ ]:
import os
for root, dirs, files in os.walk('results'):
    for f in files:
        path = os.path.join(root, f)
        print(f'{path}  ({os.path.getsize(path)/1024:.1f} KB)')